# 03 -- Estate Population Model

**Purpose (PROJECT.md Section 8.7):** deterministic tenure/insulation weighted-average headline; parameter-uncertainty Monte Carlo over PROVISIONAL inputs for P10/P50/P90; small-N street-level sensitivity; anti-correlation stress test. Week 2-3.

**Critical design decision (do not regress this):** the estate-level headline is a **deterministic weighted average**, not a population simulation -- category assignment carries no real per-dwelling stochastic variation once shares are known, so resampling N synthetic dwellings would only manufacture fake uncertainty from sampling noise. Monte Carlo is reserved for genuinely uncertain PROVISIONAL parameters and for small-N street-level realism, where the randomness is real. See PROJECT.md Section 8.5 for the full reasoning behind this.


In [1]:
import sys

print("Python executable:", sys.executable)
assert "thermal-counterfactual-gb" in sys.executable, (
    "Wrong kernel selected -- pick the 'thermal-counterfactual-gb' kernel, "
    "not a default/global one. Run setup.sh first if it doesn't exist yet."
)

import numpy as np
import polars as pl
import scipy
import matplotlib
print("polars:", pl.__version__, "| numpy:", np.__version__, "| scipy:", scipy.__version__)


Python executable: /tmp/kernelenv/thermal-counterfactual-gb/bin/python3


polars: 1.43.2 | numpy: 2.2.6 | scipy: 1.15.3


## 1. Load assumptions, Notebook 01 and Notebook 02 output

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))
import yaml
import polars as pl
import numpy as np

CONFIG_PATH = Path("../configs/tenure_insulation_assumptions.yml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

physics_01 = pl.read_parquet("../data/intermediate/01_archetype_physics.parquet")
sim_02 = pl.read_parquet("../data/intermediate/02_cold_snap_simulation.parquet")

comfort = cfg["comfort_band"]
cop = cfg["heat_pump"]["cop_at_cold_snap"]
internal_gains_w = cfg["internal_gains_w"]["point"]  # external review: house is not an empty box

peak_window_str = cfg["cold_snap_event"]["peak_window"]
start_str, end_str = peak_window_str.split("-")
peak_start = int(start_str.split(":")[0])
peak_end = int(end_str.split(":")[0])

print(physics_01)
print(f"Peak window: {peak_start:02d}:00-{peak_end:02d}:00")


shape: (3, 5)
┌────────────────┬───────────┬────────────┬─────────────────┬─────────┐
│ envelope_state ┆ h_w_per_k ┆ tau_hours  ┆ coastdown_hours ┆ peak_kw │
│ ---            ┆ ---       ┆ ---        ┆ ---             ┆ ---     │
│ str            ┆ f64       ┆ f64        ┆ f64             ┆ f64     │
╞════════════════╪═══════════╪════════════╪═════════════════╪═════════╡
│ baseline       ┆ 272.75    ┆ 36.663611  ┆ 4.999726        ┆ 2.4584  │
│ swi_only       ┆ 230.75    ┆ 43.336945  ┆ 5.982455        ┆ 2.0552  │
│ epc_c_package  ┆ 85.55     ┆ 116.890707 ┆ 18.668201       ┆ 0.66128 │
└────────────────┴───────────┴────────────┴─────────────────┴─────────┘
Peak window: 16:00-20:00


## 2. Population weights: tenure x insulation probability

In [3]:
tenure_mix = cfg["tenure_mix"]
insulation_prob = cfg["solid_wall_insulation_probability"]

# key names differ slightly between the two config blocks (tenure_mix uses
# "_retained" suffixes for social tenures) -- map explicitly rather than
# guess a naming convention silently.
tenure_to_insulation_key = {
    "local_authority_retained": "local_authority",
    "housing_association_retained": "housing_association",
    "ex_rtb_privately_rented": "ex_rtb_privately_rented",
    "ex_rtb_owner_occupied": "ex_rtb_owner_occupied",
}
tenure_names = list(tenure_to_insulation_key.keys())

weighted_insulation_prevalence = sum(
    tenure_mix[t] * insulation_prob[tenure_to_insulation_key[t]] for t in tenure_names
)
expected_prevalence = insulation_prob["weighted_current_state_prevalence"]

print(f"Weighted current-state insulation prevalence: {weighted_insulation_prevalence:.3f}")
assert abs(weighted_insulation_prevalence - expected_prevalence) < 0.005, (
    "mismatch vs configs/tenure_insulation_assumptions.yml's precomputed figure -- "
    "investigate before trusting anything downstream"
)
print("Matches the config's precomputed weighted_current_state_prevalence (0.27).")

uninsulated_share = 1 - weighted_insulation_prevalence
print(f"\n{uninsulated_share:.0%} of the estate is modelled as baseline (uninsulated) fabric today; "
      f"{weighted_insulation_prevalence:.0%} as solid-wall-insulated (swi_only).")
print()
print(
    "Caveat (external review): this is a FABRIC capability figure, not a delivered-flexibility one. "
    "The coastdown physics elsewhere in this project assumes a charged battery -- a 21C starting "
    "indoor temperature before any curtailment. Fuel poverty and prepayment-meter self-rationing "
    "concentrate in exactly the lowest-insulation-probability tenure segments modelled here (ex-RTB "
    "privately rented above all), so real occupant behaviour is plausibly ANTI-CORRELATED with fabric "
    "quality -- the streets least likely to be insulated are also plausibly least likely to be heated "
    "to 21C in the first place. If true, usable capacity across the estate is below even this 27% "
    "figure. Not modelled here (no occupant behaviour data was used); see "
    "configs/tenure_insulation_assumptions.yml, state_of_charge_behavioural_note."
)


Weighted current-state insulation prevalence: 0.270
Matches the config's precomputed weighted_current_state_prevalence (0.27).

73% of the estate is modelled as baseline (uninsulated) fabric today; 27% as solid-wall-insulated (swi_only).

Caveat (external review): this is a FABRIC capability figure, not a delivered-flexibility one. The coastdown physics elsewhere in this project assumes a charged battery -- a 21C starting indoor temperature before any curtailment. Fuel poverty and prepayment-meter self-rationing concentrate in exactly the lowest-insulation-probability tenure segments modelled here (ex-RTB privately rented above all), so real occupant behaviour is plausibly ANTI-CORRELATED with fabric quality -- the streets least likely to be insulated are also plausibly least likely to be heated to 21C in the first place. If true, usable capacity across the estate is below even this 27% figure. Not modelled here (no occupant behaviour data was used); see configs/tenure_insulation_assum

## 3. Three deterministic estate-level scenarios

Status quo (today's real tenure/insulation mix, no VPP) vs Realistic Hybrid (same mix, VPP-enrolled) vs Theoretical Ceiling (100% EPC-C, VPP-enrolled) -- the first three of PROJECT.md's four Stage D scenarios. The fourth, the Anti-Correlation Stress Test, is Section 6 below.


In [4]:
# Aggregate Notebook 02's per-hour output into peak-window summary metrics
# per envelope state -- this is the thing being weighted below.
sim_02_peak = sim_02.filter(
    (pl.col("hour_index") % 24 >= peak_start) & (pl.col("hour_index") % 24 < peak_end)
)
peak_summary = sim_02_peak.group_by("envelope_state").agg([
    pl.col("baseline_kw").mean().alias("baseline_kw_during_peak_mean"),
    pl.col("vpp_kw").mean().alias("vpp_kw_during_peak_mean"),
])
state_metrics = {row["envelope_state"]: row for row in peak_summary.iter_rows(named=True)}
print(peak_summary)


def estate_weighted(metric_col, insulated_state="swi_only", uninsulated_state="baseline"):
    """Deterministic weighted average across the four tenure segments.

    PROJECT.md Section 8.5 (Stage F): this is a Sigma-sum, not a population
    simulation -- reproducible by hand from configs/tenure_insulation_assumptions.yml
    in under 60 seconds (Traceability Mandate, Section 6).
    """
    total = 0.0
    for t in tenure_names:
        share = tenure_mix[t]
        p_ins = insulation_prob[tenure_to_insulation_key[t]]
        val_ins = state_metrics[insulated_state][metric_col]
        val_unins = state_metrics[uninsulated_state][metric_col]
        total += share * (p_ins * val_ins + (1 - p_ins) * val_unins)
    return total


status_quo_kw = estate_weighted("baseline_kw_during_peak_mean")  # today: no VPP, today's real fabric mix
realistic_hybrid_kw = estate_weighted("vpp_kw_during_peak_mean")  # today's fabric mix, VPP-enrolled
ceiling_kw = state_metrics["epc_c_package"]["vpp_kw_during_peak_mean"]  # 100% EPC-C, VPP-enrolled

print()
print(f"Status quo (today's fabric mix, no VPP):        {status_quo_kw:.2f} kW/home avg during peak")
print(f"Realistic hybrid (today's fabric mix, with VPP): {realistic_hybrid_kw:.2f} kW/home avg during peak")
print(f"Theoretical ceiling (100% EPC-C, with VPP):      {ceiling_kw:.2f} kW/home avg during peak")
print()
print(
    "Both VPP scenarios land at ~0 kW during peak for this specific event -- consistent with "
    "Notebook 02's finding that the 16:00-20:00 window isn't the coldest part of this profile, "
    "so even baseline fabric coasts through it under VPP control. The differentiator at estate "
    f"level isn't the mean avoided kW here, it's exposure: {uninsulated_share:.0%} of the estate "
    "is sitting in the fragile ~0.6C-margin state rather than the ~1.0C+ margin state -- see the "
    "stress test below for what happens when that margin gets tested harder."
)


shape: (3, 3)
┌────────────────┬──────────────────────────────┬─────────────────────────┐
│ envelope_state ┆ baseline_kw_during_peak_mean ┆ vpp_kw_during_peak_mean │
│ ---            ┆ ---                          ┆ ---                     │
│ str            ┆ f64                          ┆ f64                     │
╞════════════════╪══════════════════════════════╪═════════════════════════╡
│ baseline       ┆ 2.123866                     ┆ 0.0                     │
│ epc_c_package  ┆ 0.556351                     ┆ 0.0                     │
│ swi_only       ┆ 1.77218                      ┆ 0.0                     │
└────────────────┴──────────────────────────────┴─────────────────────────┘

Status quo (today's fabric mix, no VPP):        2.03 kW/home avg during peak
Realistic hybrid (today's fabric mix, with VPP): 0.00 kW/home avg during peak
Theoretical ceiling (100% EPC-C, with VPP):      0.00 kW/home avg during peak

Both VPP scenarios land at ~0 kW during peak for this specific even

## 4. Parameter-uncertainty Monte Carlo

In [5]:
# Parameter-uncertainty Monte Carlo (PROJECT.md Section 8.5, Stage F):
# vary only the PROVISIONAL parameters -- COP, thermal capacity C, and
# geometry -- never resample the tenure/insulation categorical mix, which
# has no real per-draw stochastic variation once its shares are fixed
# (that mistake was caught and reversed earlier in this project -- see
# PROJECT.md Section 8.5 for the reasoning).
from thermal_counterfactual_gb.physics import (
    EnvelopeState,
    heat_loss_coefficient_w_per_k,
    thermal_time_constant_hours,
    coastdown_hours,
    peak_electrical_demand_kw,
)

geometry = cfg["geometry"]
envelope_cfg = cfg["envelope_states"]
t_outdoor_design = cfg["cold_snap_event"]["design_outdoor_temp_c"]


def build_state(name):
    s = envelope_cfg[name]
    return EnvelopeState(
        wall_u_w_per_m2k=s["wall_u_w_per_m2k"],
        roof_u_w_per_m2k=s["roof_u_w_per_m2k"],
        floor_u_w_per_m2k=s["floor_u_w_per_m2k"],
        window_u_w_per_m2k=s["window_u_w_per_m2k"],
        infiltration_ach=s["infiltration_ach"],
    )


baseline_state = build_state("baseline")
epc_c_state = build_state("epc_c_package")
delta_t_peak = comfort["normal_setpoint_c"] - t_outdoor_design

rng = np.random.default_rng(seed=42)  # fixed seed -- deterministic, reproducible run
n_mc = 5000

cop_samples = rng.uniform(2.0, 3.0, n_mc)               # PROVISIONAL range around the 2.5 point estimate
c_samples = rng.uniform(7.0, 13.0, n_mc)                 # PROVISIONAL range around the 10 kWh/K point estimate
floor_area_samples = rng.uniform(60.0, 85.0, n_mc)       # PROVISIONAL range around the 70 m2 point estimate
# geometry scales as a single linear factor (fixed proportions) -- a documented
# simplification, not a claim that every dimension scales identically in reality
scale = floor_area_samples / geometry["floor_area_m2"]

avoided_kw_samples = np.zeros(n_mc)
coastdown_epc_c_samples = np.zeros(n_mc)

for i in range(n_mc):
    wall_a = geometry["opaque_wall_area_m2"] * scale[i]
    roof_a = geometry["roof_area_m2"] * scale[i]
    floor_a = geometry["ground_floor_area_m2"] * scale[i]
    window_a = geometry["window_area_m2"] * scale[i]
    volume = geometry["heated_volume_m3"] * scale[i]

    h_baseline_i = heat_loss_coefficient_w_per_k(baseline_state, wall_a, roof_a, floor_a, window_a, volume)
    h_epc_c_i = heat_loss_coefficient_w_per_k(epc_c_state, wall_a, roof_a, floor_a, window_a, volume)

    peak_kw_baseline_i = peak_electrical_demand_kw(h_baseline_i, delta_t_peak, cop_samples[i], internal_gains_w=internal_gains_w)
    peak_kw_epc_c_i = peak_electrical_demand_kw(h_epc_c_i, delta_t_peak, cop_samples[i], internal_gains_w=internal_gains_w)
    avoided_kw_samples[i] = peak_kw_baseline_i - peak_kw_epc_c_i

    tau_epc_c_i = thermal_time_constant_hours(c_samples[i], h_epc_c_i)
    coastdown_epc_c_samples[i] = coastdown_hours(
        tau_epc_c_i, comfort["preheat_ceiling_c"], comfort["minimum_c"], t_outdoor_design,
        h_w_per_k=h_epc_c_i, internal_gains_w=internal_gains_w,
    )

avoided_p10, avoided_p50, avoided_p90 = np.percentile(avoided_kw_samples, [10, 50, 90])
coast_p10, coast_p50, coast_p90 = np.percentile(coastdown_epc_c_samples, [10, 50, 90])

print(f"Peak kW avoided per home, baseline->EPC-C (design temp {t_outdoor_design}C), n={n_mc} draws:")
print(f"  P10={avoided_p10:.2f} kW   P50={avoided_p50:.2f} kW   P90={avoided_p90:.2f} kW")
print(f"EPC-C coastdown hours at design temp, n={n_mc} draws:")
print(f"  P10={coast_p10:.1f} h   P50={coast_p50:.1f} h   P90={coast_p90:.1f} h")


Peak kW avoided per home, baseline->EPC-C (design temp -3C), n=5000 draws:
  P10=1.53 kW   P50=1.87 kW   P90=2.30 kW
EPC-C coastdown hours at design temp, n=5000 draws:
  P10=13.1 h   P50=17.9 h   P90=23.5 h


## 4b. Ex-Right-to-Buy split sensitivity (simple two-point check, not folded into the Monte Carlo)

In [6]:
# The ex-RTB rented/owner split (0.15 / 0.20 of the estate) is derived from
# NEF's imprecise ">4 in 10" finding, not a precise published ratio -- a
# simple two-point sensitivity is more honest than folding a made-up
# distribution shape into the Monte Carlo above.
ex_rtb_total = tenure_mix["ex_rtb_privately_rented"] + tenure_mix["ex_rtb_owner_occupied"]  # 0.35, well-sourced
rented_fraction_low, rented_fraction_high = 0.35, 0.50  # bracketing NEF's 0.42 point estimate

for label, rented_fraction in [("low (35% of ex-RTB rented)", rented_fraction_low), ("point estimate (42%)", 0.42), ("high (50% of ex-RTB rented)", rented_fraction_high)]:
    rented_share = ex_rtb_total * rented_fraction
    owner_share = ex_rtb_total * (1 - rented_fraction)
    prevalence = (
        tenure_mix["local_authority_retained"] * insulation_prob["local_authority"]
        + tenure_mix["housing_association_retained"] * insulation_prob["housing_association"]
        + rented_share * insulation_prob["ex_rtb_privately_rented"]
        + owner_share * insulation_prob["ex_rtb_owner_occupied"]
    )
    print(f"  ex-RTB rented fraction {label:32s} -> weighted insulation prevalence {prevalence:.3f}")

print("\nThe ex-RTB split barely moves the headline (rented and owner-occupied insulation "
      "probabilities are close, 10% vs 11%) -- worth knowing it's not a load-bearing uncertainty here.")


  ex-RTB rented fraction low (35% of ex-RTB rented)       -> weighted insulation prevalence 0.271
  ex-RTB rented fraction point estimate (42%)             -> weighted insulation prevalence 0.270
  ex-RTB rented fraction high (50% of ex-RTB rented)      -> weighted insulation prevalence 0.270

The ex-RTB split barely moves the headline (rented and owner-occupied insulation probabilities are close, 10% vs 11%) -- worth knowing it's not a load-bearing uncertainty here.


## 5. Small-N street-level sensitivity

In [7]:
# Small-N street-level sensitivity: this IS a legitimate use of random
# sampling (unlike the estate-wide population, a specific ~30-home street
# (a real LV feeder typically serves ~100-300 homes -- external review
# correctly flagged "feeder" as the wrong term for a 30-home group) has a
# realised composition that really can deviate from the estate average by
# chance -- a DNO plans for a specific street or sub-feeder, not an
# abstract blend).
feeder_size = 30
n_feeders = 5000

tenure_probs_arr = np.array([tenure_mix[t] for t in tenure_names])
insulation_probs_arr = np.array([insulation_prob[tenure_to_insulation_key[t]] for t in tenure_names])

rng2 = np.random.default_rng(seed=7)

feeder_insulation_prevalence = np.zeros(n_feeders)
for f in range(n_feeders):
    tenure_draw = rng2.choice(len(tenure_names), size=feeder_size, p=tenure_probs_arr)
    insulated_draw = rng2.random(feeder_size) < insulation_probs_arr[tenure_draw]
    feeder_insulation_prevalence[f] = insulated_draw.mean()

feeder_p10, feeder_p50, feeder_p90 = np.percentile(feeder_insulation_prevalence, [10, 50, 90])
feeder_min = feeder_insulation_prevalence.min()

print(f"Simulated {n_feeders} random {feeder_size}-home streets (sub-feeder samples) drawn from the estate-wide tenure/insulation mix:")
print(f"  P10={feeder_p10:.0%}   P50={feeder_p50:.0%}   P90={feeder_p90:.0%}   worst observed={feeder_min:.0%} insulated")
print(f"\nThe estate-wide expectation is {weighted_insulation_prevalence:.0%} insulated, but a specific "
      f"street can look very different by chance alone -- some streets in this simulation had as little "
      f"as {feeder_min:.0%} insulated stock. A DNO reading only the estate-average figure would misjudge "
      f"what a specific street can actually deliver. Real tenure is also spatially autocorrelated (whole "
      f"streets were sold under Right to Buy together) in a way this random draw does not capture, so "
      f"this range is best read as a LOWER bound on real-world street-to-street variation.")


Simulated 5000 random 30-home streets (sub-feeder samples) drawn from the estate-wide tenure/insulation mix:
  P10=17%   P50=27%   P90=37%   worst observed=0% insulated

The estate-wide expectation is 27% insulated, but a specific street can look very different by chance alone -- some streets in this simulation had as little as 0% insulated stock. A DNO reading only the estate-average figure would misjudge what a specific street can actually deliver. Real tenure is also spatially autocorrelated (whole streets were sold under Right to Buy together) in a way this random draw does not capture, so this range is best read as a LOWER bound on real-world street-to-street variation.


## 6. Anti-Correlation Stress Test

In [8]:
# Anti-Correlation Stress Test (PROJECT.md Section 2.4 / 8.5): does the
# "hidden battery" get smallest exactly when the grid needs it most?
#
# Two things are varied together, deliberately, because that's the
# correlation the test exists to probe:
#   (a) how much colder than the real recorded event would it take before
#       baseline fabric starts breaching the comfort floor during the peak
#       window at all (not just on average, as Notebook 02 checked)?
#   (b) at that point, what fraction of dwellings are simultaneously
#       forced to resume heating -- on the diversified estate average,
#       versus a street concentrated in the worst-insulated tenure?
from thermal_counterfactual_gb.physics import simulate_vpp_control_hours

outdoor_base = (
    sim_02.filter(pl.col("envelope_state") == "baseline")
    .sort("hour_index")["outdoor_temp_c"]
    .to_numpy()
)
h_baseline = physics_01.filter(pl.col("envelope_state") == "baseline")["h_w_per_k"].item()
tau_baseline = physics_01.filter(pl.col("envelope_state") == "baseline")["tau_hours"].item()

peak_hours_per_week = np.arange(len(outdoor_base)) % 24
peak_mask = (peak_hours_per_week >= peak_start) & (peak_hours_per_week < peak_end)

deltas_c = [0, -1, -2, -3, -4, -5, -6, -7, -8, -10, -12]
sweep_rows = []
for d in deltas_c:
    shifted = outdoor_base + d
    indoor, heating_on, kw = simulate_vpp_control_hours(
        tau_baseline, h_baseline, shifted,
        comfort["normal_setpoint_c"], comfort["preheat_ceiling_c"], comfort["minimum_c"],
        cop, peak_start, peak_end, internal_gains_w=internal_gains_w,
    )
    resumed_hours_in_peak = int(heating_on[peak_mask].sum())
    outdoor_min_in_peak = float(shifted[peak_mask].min())
    sweep_rows.append({
        "delta_c": d,
        "resumed_hours_in_peak": resumed_hours_in_peak,
        "min_indoor_c": float(indoor[peak_mask].min()),
        "outdoor_min_c_in_peak": outdoor_min_in_peak,
    })

for row in sweep_rows:
    print(f"  outdoor shift {row['delta_c']:+d}C   resumed-heating hours during peak: {row['resumed_hours_in_peak']:2d} / {int(peak_mask.sum())}   min indoor: {row['min_indoor_c']:.2f}C")

# find the first (least-severe) delta where any breach occurs
breach_row = next((row for row in sweep_rows if row["resumed_hours_in_peak"] > 0), None)
breach_delta = breach_row["delta_c"] if breach_row is not None else None
print()
max_delta_tested = deltas_c[-1]
if breach_row is None:
    print(f"No breach found within the tested range -- baseline fabric's margin in this event holds up "
          f"to at least {max_delta_tested}C colder than the recorded hourly profile.")
else:
    print(f"Baseline fabric starts breaching the comfort floor during peak once the event is "
          f"{breach_delta}C colder than the recorded hourly profile "
          f"(outdoor min in peak window at that point: {breach_row['outdoor_min_c_in_peak']:.1f}C).")


  outdoor shift +0C   resumed-heating hours during peak:  0 / 28   min indoor: 19.76C
  outdoor shift -1C   resumed-heating hours during peak:  0 / 28   min indoor: 19.66C
  outdoor shift -2C   resumed-heating hours during peak:  0 / 28   min indoor: 19.55C
  outdoor shift -3C   resumed-heating hours during peak:  0 / 28   min indoor: 19.45C
  outdoor shift -4C   resumed-heating hours during peak:  0 / 28   min indoor: 19.35C
  outdoor shift -5C   resumed-heating hours during peak:  0 / 28   min indoor: 19.24C
  outdoor shift -6C   resumed-heating hours during peak:  0 / 28   min indoor: 19.14C
  outdoor shift -7C   resumed-heating hours during peak:  0 / 28   min indoor: 19.04C
  outdoor shift -8C   resumed-heating hours during peak:  3 / 28   min indoor: 19.00C
  outdoor shift -10C   resumed-heating hours during peak:  6 / 28   min indoor: 19.00C
  outdoor shift -12C   resumed-heating hours during peak:  7 / 28   min indoor: 19.00C

Baseline fabric starts breaching the comfort floor 

## 6b. Concentrated-tenure street exposure at the breach threshold

In [9]:
# At the point identified above (an actual breach if the sweep found one,
# otherwise the coldest case tested, clearly labelled as such), compare
# who's exposed: the diversified estate average versus a STREET (a real LV
# feeder typically serves ~100-300 homes -- external review correctly
# flagged "feeder" as the wrong term for a 30-home group; this project
# calls it a street or sub-feeder sample throughout) concentrated in the
# worst-insulated tenure (ex-RTB privately rented, only 10% insulated).
diversified_uninsulated_share = uninsulated_share  # 1 - weighted prevalence, computed above
worst_tenure_insulation = insulation_prob["ex_rtb_privately_rented"]
concentrated_uninsulated_share = 1 - worst_tenure_insulation

reference_row = breach_row if breach_row is not None else sweep_rows[-1]
reference_outdoor_min = reference_row["outdoor_min_c_in_peak"]
reference_delta = reference_row["delta_c"]

if breach_row is not None:
    print(f"At the breach point found above (outdoor shift {reference_delta:+d}C, "
          f"outdoor min {reference_outdoor_min:.1f}C in the peak window), who is exposed:")
else:
    print(f"No breach occurred even at the coldest case tested (outdoor shift {reference_delta:+d}C, "
          f"outdoor min {reference_outdoor_min:.1f}C in the peak window). The comparison below therefore "
          f"uses that coldest tested case as an illustrative stress reference, not an actual observed breach.")

print()
print(f"Share of dwellings in the fragile baseline (uninsulated) state, and so exposed once a real "
      f"breach does occur:")
print(f"  Diversified estate average:                {diversified_uninsulated_share:.0%}")
print(f"  Street concentrated in ex-RTB rented stock: {concentrated_uninsulated_share:.0%}")

feeder_dwellings = 30
diversified_breaching = round(diversified_uninsulated_share * feeder_dwellings)
concentrated_breaching = round(concentrated_uninsulated_share * feeder_dwellings)

# Steady-state resume-heating demand per home, evaluated at the reference
# outdoor temperature identified above -- not at the original -3C design
# point, so this stays consistent with whatever case is actually being described.
# Nets off internal_gains_w for the same reason every other heating-power
# calculation in this project does -- this was found missing it during a
# later review pass and fixed here rather than left inconsistent.
from thermal_counterfactual_gb.physics import net_heating_power_kw
per_home_resume_kw = net_heating_power_kw(
    h_baseline, comfort["minimum_c"], reference_outdoor_min, cop, internal_gains_w=internal_gains_w
)

diversified_spike_kw = diversified_breaching * per_home_resume_kw
concentrated_spike_kw = concentrated_breaching * per_home_resume_kw

print()
print(f"On a {feeder_dwellings}-home street, at that reference point, resume-heating would kick in "
      f"simultaneously for roughly:")
print(f"  Diversified street:   {diversified_breaching}/{feeder_dwellings} homes -> ~{diversified_spike_kw:.1f} kW coincident spike")
print(f"  Concentrated street:  {concentrated_breaching}/{feeder_dwellings} homes -> ~{concentrated_spike_kw:.1f} kW coincident spike")
print()
print(
    "This is the anti-correlation risk in concrete terms: the flexibility a VPP or DNO might count "
    "on from fabric-driven coasting is smallest exactly when the weather is coldest -- and smallest "
    "again on the streets where the underlying tenure mix is least insulated, disproportionately the "
    "ex-Right-to-Buy privately rented streets this project already flagged as sitting outside a "
    "housing association's direct retrofit reach. This notebook has NOT shown that these "
    "low-insulation streets spatially coincide with a DNO's actual constrained feeders (EV uptake, "
    "electrified heat, and weak LV assets could concentrate anywhere on the network, independent of "
    "tenure history) -- that overlap is a plausible, testable hypothesis worth checking against real "
    "network data, not a finding this notebook has established. This is illustrative arithmetic, not "
    "a calibrated network model -- it uses the same simple resume-heating physics as the rest of the "
    "notebook, scaled up, not a real feeder or network topology."
)
print()
print(
    "One more thing this arithmetic leaves out: the coincident spike above is a snapshot of the "
    "moment heating resumes, not the end of the story. Every hour of curtailment that ends in resumed "
    "heating displaces that demand to the recovery period rather than deleting it -- the 'cold pickup' "
    "load when many homes reheat together is itself a coincident-demand event, and if cold snaps recur "
    "on consecutive days, the same homes may not fully recover comfort between events. That's a "
    "comfort-and-equity concern for exactly the vulnerable households this project is most concerned "
    "about, not just a grid one, and it is not quantified here."
)


At the breach point found above (outdoor shift -8C, outdoor min -9.4C in the peak window), who is exposed:

Share of dwellings in the fragile baseline (uninsulated) state, and so exposed once a real breach does occur:
  Diversified estate average:                73%
  Street concentrated in ex-RTB rented stock: 90%

On a 30-home street, at that reference point, resume-heating would kick in simultaneously for roughly:
  Diversified street:   22/30 homes -> ~64.6 kW coincident spike
  Concentrated street:  27/30 homes -> ~79.2 kW coincident spike

This is the anti-correlation risk in concrete terms: the flexibility a VPP or DNO might count on from fabric-driven coasting is smallest exactly when the weather is coldest -- and smallest again on the streets where the underlying tenure mix is least insulated, disproportionately the ex-Right-to-Buy privately rented streets this project already flagged as sitting outside a housing association's direct retrofit reach. This notebook has NOT shown

## 7. Save to `data/intermediate/`

In [10]:
summary = pl.DataFrame([
    {"scenario": "status_quo_no_vpp", "kw_per_home_during_peak": status_quo_kw},
    {"scenario": "realistic_hybrid_with_vpp", "kw_per_home_during_peak": realistic_hybrid_kw},
    {"scenario": "theoretical_ceiling_epc_c_vpp", "kw_per_home_during_peak": ceiling_kw},
]).with_columns(
    pl.lit(weighted_insulation_prevalence).alias("weighted_insulation_prevalence"),
    pl.lit(avoided_p50).alias("mc_avoided_kw_p50"),
    pl.lit(avoided_p10).alias("mc_avoided_kw_p10"),
    pl.lit(avoided_p90).alias("mc_avoided_kw_p90"),
    pl.lit(feeder_p50).alias("feeder_insulation_prevalence_p50"),
    pl.lit(feeder_min).alias("feeder_insulation_prevalence_worst_observed"),
    pl.lit(breach_delta if breach_delta is not None else float("nan")).alias("stress_test_breach_delta_c"),
)

out_path = Path("../data/intermediate/03_estate_population_model.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.write_parquet(out_path)
print(f"Wrote {out_path} ({summary.height} rows)")
summary


Wrote ../data/intermediate/03_estate_population_model.parquet (3 rows)


scenario,kw_per_home_during_peak,weighted_insulation_prevalence,mc_avoided_kw_p50,mc_avoided_kw_p10,mc_avoided_kw_p90,feeder_insulation_prevalence_p50,feeder_insulation_prevalence_worst_observed,stress_test_breach_delta_c
str,f64,f64,f64,f64,f64,f64,f64,i32
"""status_quo_no_vpp""",2.028806,0.2703,1.867537,1.527288,2.299703,0.266667,0.0,-8
"""realistic_hybrid_with_vpp""",0.0,0.2703,1.867537,1.527288,2.299703,0.266667,0.0,-8
"""theoretical_ceiling_epc_c_vpp""",0.0,0.2703,1.867537,1.527288,2.299703,0.266667,0.0,-8
